# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and performing basic analysis on the FAIR² dataset using the `mlcroissant` library. All references to entities (record sets, fields, columns) use their `@id` identifiers, as required for interoperability and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print("Description:")
print(metadata.description)


## 2. Data Overview
Explore the available record sets, their fields, and how to refer to them by `@id`.


In [ ]:
# Print all record sets and their @id values
print("All available record sets and fields (by @id):\n")

for record_set in dataset.metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  name: {record_set.name}")
    print(f"  description: {getattr(record_set, 'description', 'No description')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.id} ({field.name})  [dataType: {getattr(field, 'data_type', 'unknown')}]" )
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all RecordSet @id values; select all by default
record_sets = [r.id for r in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Load all records for the record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print available columns for each DataFrame
print("Record set DataFrame columns:")
for rsid, df in dataframes.items():
    print(f"- {rsid}: {df.columns.tolist()}")

# Display first few rows of the primary record set (pick the largest or primary one)
# For this dataset, look for clinical/tabular data, typically it's the first one
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id:
    display_df = dataframes[primary_record_set_id]
    print(f"\nHead of record set: {primary_record_set_id}")
    display_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All field references use their `@id`.

In [ ]:
# For demonstration, select a numeric field and a group field by @id based on the earlier overview
# (Replace with the relevant field @id's as shown in section 2's prints)
# Example placeholders:
# numeric_field_id = 'http://senscience.ai/age_at_second_crc'  # Replace as appropriate
# group_field_id = 'http://senscience.ai/sex'                 # Replace as appropriate

primary_record_set_id = record_sets[0] if record_sets else None
df = dataframes[primary_record_set_id]

# List all available columns to help select correct @id
print(f"Available columns for {primary_record_set_id}:")
pprint.pprint(list(df.columns))

# Choose field @id's that exist (example: age at diagnosis, sex, etc.)
numeric_field_id = None
group_field_id = None
# Try to auto-detect a suitable numeric field and group field
for col in df.columns:
    # Choose the first int/float-like column as numeric, and a likely categorical as group
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    if group_field_id is None and (
        'sex' in col.lower() or 'gender' in col.lower() or 'site' in col.lower() or 'location' in col.lower()
    ):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

print(f"\nUsing numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# If none found, you may need to manually enter the '@id' from earlier output
if numeric_field_id is None or group_field_id is None:
    print("Please update 'numeric_field_id' and 'group_field_id' to match one of the available columns above.")

# Continue if found
if numeric_field_id and group_field_id:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field and show means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id and group_field_id and not df.empty:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a publicly FAIR²-compliant clinical dataset using `mlcroissant`. By referencing all entities by their `@id` values, users can reproducibly extract, process, analyze, and visualize complex tabular biomedical data. The same approach is suitable for any data package provided in Croissant format—with meaningful results dependent on the field definitions found via each entity's `@id` in the schema.

For more advanced usage, consider joining multiple record sets, implementing domain-specific filters, or adapting the visualization step to publication quality. Always refer to the Croissant schema documentation for full entity and attribute details.
